# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [2]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


## Ejemplo

In [3]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [4]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [5]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [13]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [8]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.118853,0.140983,0.982989,0.772973,0.982407,0.018471,0.000642,0.007697,0.000754
TensorizedQDA,0.098785,0.134035,0.434110,0.542711,0.982593,0.018471,0.000653,0.012123,0.000191


In [9]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.118853,0.982989,0.982407
TensorizedQDA,0.098785,0.434110,0.982593


In [10]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.118853,0.140983,0.982989,0.772973,0.982407,0.018471,0.000642,0.007697,0.000754,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.098785,0.134035,0.434110,0.542711,0.982593,0.018471,0.000653,0.012123,0.000191,1.203148,2.264378,1.0,0.634912


In [11]:
# volvemos a subsetear columnas
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.118853,0.982989,0.982407,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.098785,0.434110,0.982593,1.203148,2.264378,1.0,0.634912


In [18]:
# Levantamos los datasets Iris, Letters, Penguins y Wine
X_full_iris, y_full_iris = get_iris_dataset()
# X_full_letters, y_full_letters = get_letters_dataset()
# X_full_penguins, y_full_penguins = get_penguins_dataset()
X_full_wine, y_full_wine = get_wine_dataset()

X_full_iris.shape, y_full_iris.shape
# X_full_letters.shape, y_full_letters.shape
# X_full_penguins.shape, y_full_penguins.shape
X_full_wine.shape, y_full_wine.shape

((178, 13), (178, 1))

In [19]:
# Encodeamos a número las clases
y_full_iris_encoded = label_encode(y_full_iris)
y_full_wine_encoded = label_encode(y_full_wine)

y_full_iris[:5], y_full_iris_encoded[:5]
y_full_wine[:5], y_full_wine_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [21]:
benchmark_iris = Benchmark(
    X_full_iris, y_full_iris_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

benchmark_wine = Benchmark(
    X_full_wine, y_full_wine_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [22]:
# Benchmark de los modemos en el dataset Iris
to_bench = [QDA, TensorizedQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    benchmark_iris.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [23]:
for model in to_bench:
    benchmark_wine.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [24]:
summary_iris = benchmark_iris.summary(baseline='QDA')
summary_iris[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.111888,0.751391,0.970667,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.109162,0.271403,0.971111,1.024963,2.768545,1.011858,0.878836
QDA_Chol1,0.134602,0.499665,0.972889,0.831250,1.503791,0.964312,0.947228
QDA_Chol2,0.108331,1.069944,0.972222,1.032830,0.702272,0.993261,0.902874
QDA_Chol3,0.108577,0.467566,0.977333,1.030490,1.607029,0.996313,1.020702


In [25]:
summary_wine = benchmark_wine.summary(baseline='QDA')
summary_wine[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.118565,0.990413,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.123496,0.440356,0.982593,0.960068,2.249116,1.000000,0.630729
QDA_Chol1,0.145289,0.580786,0.985741,0.816063,1.705296,1.023799,0.973355
QDA_Chol2,0.114870,1.281227,0.983333,1.032167,0.773019,1.019683,0.932698
QDA_Chol3,0.115297,0.579343,0.986111,1.028349,1.709547,1.032134,0.991867
